# Using the Fermi Data Platform

This is a demonstration of reading and writing data and metadata on the Fermi Data Platform (FDP). You will learn to
- Browse and download public data using WebDAV (HTTPS)
- Look at metadata
- Authenticate
- Browse and download private data
- Write and delete files
- Add and update metadata

This notebook serves as a simple demonstration of how to use FDP. More information on getting an account, other data access methods, and handling metadata are covered in the [FDP documentation](https://github.com/FNAL-SCD/fdpdocs). For more information on how to find data in FDP via AmSC, please see the AmSC resources.

## Setup

Install this repository with the `tutorial` extra, which includes everything this notebook needs:

```bash
pip install ".[tutorial]"
```

## 1. Browsing and downloading public data

Anything under `/amsc/public/<project>` is publicly readable without credentials.

This notebook shows how to access files using WebDAV/HTTPS. For other access methods (XRootD, Globus), please see the [FDP documentation](https://github.com/FNAL-SCD/fdpdocs/accessing_datasets.html).

### List files in a directory

Modify the `DIRECTORY` variable to the directory you would like to inspect. Directories are shown with a trailing `/`.

In [ ]:
DIRECTORY = 
# DIRECTORY = "/amsc/public/fdptest"

import requests
import xml.etree.ElementTree as ET

AMSC_WEBDAV = "https://amsc.fnal.gov:2880"
DAV = "{DAV:}"

def ls(remote_path, token=None):
    """List the files/directories directly under remote_path. Directories get a trailing '/'."""
    remote_path = remote_path.rstrip("/")
    headers = {"Depth": "1"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    # A plain GET on a WebDAV directory returns an HTML page meant for browsers.
    # Listing files cleanly instead means sending a PROPFIND request, which returns XML we can parse.
    resp = requests.request("PROPFIND", f"{AMSC_WEBDAV}{remote_path}/", headers=headers)
    resp.raise_for_status()

    names = []
    for response in ET.fromstring(resp.content).findall(f"{DAV}response"):
        href = response.findtext(f"{DAV}href").rstrip("/")
        if href.endswith(remote_path):
            continue
        name = href.rsplit("/", 1)[-1]
        prop = response.find(f"{DAV}propstat/{DAV}prop")
        is_dir = prop.find(f"{DAV}resourcetype/{DAV}collection") is not None
        names.append(name + "/" if is_dir else name)
    return names

ls(DIRECTORY)

### Download a file

Modify `FDP_FILE` to the file you would like to download from FDP and `DESTINATION` to the location you would like to download it.

In [ ]:
FDP_FILE = 
DESTINATION = 
# FDP_FILE = "/amsc/public/fdptest/README"
# DESTINATION = "./README"

def download_public_file(remote_path, local_path, chunk_size=1 << 20):
    """Download a file that lives under /amsc/public/..."""
    with requests.get(f"{AMSC_WEBDAV}{remote_path}", stream=True) as resp:
        resp.raise_for_status()
        with open(local_path, "wb") as out:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                out.write(chunk)
    return local_path

download_public_file(FDP_FILE, DESTINATION)

### Stream a file

For formats that support random access (HDF5, Parquet, ...), `fsspec` can open a remote URL as a file-like object so a library like `h5py` only pulls the bytes it touches without a full download.

Modify `STREAM_FILE` to a file you would like to read parts of without downloading.

In [ ]:
STREAM_FILE = 
# STREAM_FILE = "/amsc/public/treasure/aoj/data/RunG_batch0.h5"

import fsspec
import h5py

def stream_h5(remote_path):
    """Print the top-level structure of a remote HDF5 file without downloading it."""
    with fsspec.open(f"{AMSC_WEBDAV}{remote_path}", "rb") as stream:
        with h5py.File(stream, "r") as f:
            for key in f.keys():
                print(key)

stream_h5(STREAM_FILE)

## 2. Looking up metadata in MetaCat

Files on FDP are also registered in [MetaCat](https://fermitools.github.io/metacat/), which lets you find files by description instead of needing to already know the path. Metadata can be queried without logging in.

### List datasets in a namespace

Modify `NAMESPACE_PATTERN` to the namespace you would like to list datasets from. The namespace is usually the project name.

In [ ]:
NAMESPACE_PATTERN = 
# NAMESPACE_PATTERN = "treasure*"

from metacat.webapi import MetaCatClient

METACAT_URL = "https://metacat.fnal.gov:9443/amsc_meta_prod/app"
mc = MetaCatClient(METACAT_URL)

def show_datasets(namespace_pattern):
    for ds in mc.list_datasets(namespace_pattern=namespace_pattern, with_counts=True):
        print(f"{ds['namespace']}:{ds['name']}  ({ds.get('file_count', '?')} files)")

show_datasets(NAMESPACE_PATTERN)

### List files in a dataset

Modify `DATASET` to the dataset you would like to inspect. It takes the form `namespace:dataset`.

In [ ]:
DATASET = 
# DATASET = "treasure:collide-1m"

def show_dataset_files(dataset_did):
    for f in mc.get_dataset_files(dataset_did):
        print(f"{f['name']}  {f['size']:,} bytes")

show_dataset_files(DATASET)

### Show file metadata

Modify `FILENAME` to the file you would like to see the details of. It takes the form `namespace:filename`.

In [ ]:
FILENAME = 
# FILENAME = "treasure:DYJetsToLL_13TeV-madgraphMLM-pythia8-NEVENT10000-RS10000016.parquet"

def show_file_metadata(file_did):
    record = mc.get_file(did=file_did, with_metadata=True)
    if record is None:
        print(f"No such file: {file_did}")
        return

    metadata = record["metadata"]
    width = max(len(key) for key in metadata)
    for key, value in sorted(metadata.items()):
        print(f"{key:{width}} : {value}")

show_file_metadata(FILENAME)

## 3. Authenticating

Data in the private area (located in `/amsc/<project>/...` instead of `/amsc/public/...`) and any write access require a bearer token from `htgettoken`. Run one of the following in a terminal **before** starting Jupyter:

```bash
htgettoken -a htvaultprod.fnal.gov -i amsc -r <ROLENAME>read     # read access
htgettoken -a htvaultprod.fnal.gov -i amsc -r <ROLENAME>write    # write access
```

Swap `<ROLENAME>` for your project's role. The token expires, so you'll need to re-run this periodically rather than once.

The `write` token also allows for reading the private areas. If you have read access but not write access, you will be able to read private data but you will be unable to complete the sections that involve write actions.

In [ ]:
import os
import base64
import json
import time

def _token_expiry(token):
    """Decode the exp claim from a JWT without verifying its signature."""
    try:
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        return json.loads(base64.urlsafe_b64decode(payload))["exp"]
    except Exception:
        return None

def current_token(verbose=True):
    if "BEARER_TOKEN" in os.environ:
        source = "$BEARER_TOKEN"
        tok = os.environ["BEARER_TOKEN"]
    else:
        uid = os.getuid()
        candidates = []
        if "BEARER_TOKEN_FILE" in os.environ:
            candidates.append(os.environ["BEARER_TOKEN_FILE"])
        if "XDG_RUNTIME_DIR" in os.environ:
            candidates.append(os.path.join(os.environ["XDG_RUNTIME_DIR"], f"bt_u{uid}"))
        candidates.append(f"/tmp/bt_u{uid}")

        tok = None
        for path in candidates:
            if os.path.exists(path):
                source = path
                with open(path) as f:
                    tok = f.read().strip()
                break

        if tok is None:
            raise FileNotFoundError(
                f"No bearer token found (checked {candidates}). Run htgettoken in a terminal first."
            )

    exp = _token_expiry(tok)
    if exp is not None and exp < time.time():
        raise RuntimeError(
            f"Token from {source} expired at {time.ctime(exp)}. Re-run htgettoken and restart the kernel."
        )

    if verbose:
        msg = f"Using token from {source}"
        if exp is not None:
            msg += f" (valid until {time.ctime(exp)})"
        print(msg)
    return tok

token = current_token()

## 4. Reading and writing project data

Same calls as above but with an `Authorization: Bearer` header attached. Note that dCache refuses to overwrite an existing file, so delete it first if you need to replace it.

### Download private file

Modify the variables below to a directory and file in a private area on FDP that you would like to list and download.

In [ ]:
PRIVATE_DIRECTORY = 
PRIVATE_FILE = 
LOCAL_DESTINATION = 
# PRIVATE_DIRECTORY = "/amsc/fdptest"
# PRIVATE_FILE = "/amsc/fdptest/README"
# LOCAL_DESTINATION = "./README"

def auth_headers(token):
    return {"Authorization": f"Bearer {token}"}

def download_project_file(remote_path, local_path, token, chunk_size=1 << 20):
    with requests.get(f"{AMSC_WEBDAV}{remote_path}", headers=auth_headers(token), stream=True) as resp:
        resp.raise_for_status()
        with open(local_path, "wb") as out:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                out.write(chunk)
    return local_path

ls(PRIVATE_DIRECTORY, token=token)
download_project_file(PRIVATE_FILE, LOCAL_DESTINATION, token)

### Upload a file

This is the same whether uploading to a private or public area in FDP. `FDP_DESTINATION` is the full destination path, including the filename (`PUT` on a directory path instead of a file path fails with a `405 Method Not Allowed`).

In [ ]:
FILE_TO_UPLOAD = 
FDP_DESTINATION = 
# FILE_TO_UPLOAD = "example"
# FDP_DESTINATION = "/amsc/fdptest/example"

def upload_file(local_path, remote_path, token):
    with open(local_path, "rb") as f:
        resp = requests.put(f"{AMSC_WEBDAV}{remote_path}", headers=auth_headers(token), data=f)
    resp.raise_for_status()

upload_file(FILE_TO_UPLOAD, FDP_DESTINATION, token)
upload_file("testfile", "/amsc/fdptest/testfile", token)
ls(PRIVATE_DIRECTORY, token=token)

### Delete a file

Modify `FILE_TO_DELETE` to select the file you would like to delete from FDP.

In [ ]:
FILE_TO_DELETE = 
# FILE_TO_DELETE = "/amsc/fdptest/example"

def delete_file(remote_path, token):
    resp = requests.delete(f"{AMSC_WEBDAV}{remote_path}", headers=auth_headers(token))
    if resp.status_code == 404:
        print(f"{remote_path} is already gone")
        return
    resp.raise_for_status()

delete_file(FILE_TO_DELETE, token)
ls(PRIVATE_DIRECTORY, token=token)

## 5. Adding and updating metadata

While no authentication is required for reading anything in MetaCat, a token with a write role is required for write operations in your project namespace.

Run the following cell to initialize and login to the MetaCat client.

In [ ]:
import os
from metacat.webapi import MetaCatClient

mc = MetaCatClient("https://metacat.fnal.gov:9443/amsc_meta_prod/app")

username = os.environ["USER"]
mc.login_token(username, token)

### Create a namespace and dataset

Files are declared into a dataset, which lives in a namespace. Here both are created automatically if they don't already exist. Modify `NAMESPACE` and `DATASET` below.

Please see the [documentation](https://fnal-scd.github.io/fdpdocs/metadata.html#metadata-format) for information on naming conventions and requirements of metadata fields in FDP.

In [ ]:
NAMESPACE = 
DATASET = 
DATASET_METADATA = 
# NAMESPACE = "fdptest"
# DATASET = "testds"
# DATASET_METADATA = {
    "AmSC.common.type": "scientificWork",
    "AmSC.common.display_name": f"{DATASET}",
    "AmSC.common.description": "This is a test dataset",
    "AmSC.common.location": "https://amsc.fnal.gov:2880/amsc/fdptest"
}

def ensure_namespace(name):
    if mc.get_namespace(name) is None:
        mc.create_namespace(name)
        print(f"Created namespace {name}")
    else:
        print(f"Namespace {name} already exists")

def ensure_dataset(namespace, dataset, metadata=None):
    did = f"{namespace}:{dataset}"
    if mc.get_dataset(did) is None:
        mc.create_dataset(did, metadata=DATASET_METADATA)
        print(f"Created dataset {did}")
    else:
        print(f"Dataset {did} already exists")

ensure_namespace(NAMESPACE)
ensure_dataset(NAMESPACE, DATASET)

### Declare a file

Declaring a file to MetaCat registers its size, checksum, and metadata, and adds it to a dataset. For declaring a whole directory at once, see [generate_metadata_remote.py](https://github.com/FNAL-SCD/fdpdocs/blob/main/examples/generate_metadata_remote.py) and [declare_to_metacat.py](https://github.com/FNAL-SCD/fdpdocs/blob/main/examples/declare_to_metacat.py) in this repository.

Modify the variables below to describe a file you've already uploaded to FDP (see section 4). The size and checksum are looked up from dCache.

Note that the Globus UUID listed here is the general FDP read-only public collection. This is the correct UUID to use in the metadata in most cases. Substitute your project's private collection UUID in for data in the private areas.

In [ ]:
FILE_TO_DECLARE = 
FDP_LOCATION = 
# FILE_TO_DECLARE = "testfile"
# FDP_LOCATION = "/amsc/fdptest/testfile"

AMSC_NAMESPACE_API = "https://amsc.fnal.gov:3880/api/v1/namespace"
FDP_GLOBUS_UUID = "73edb319-d876-4143-b5fc-5a47c61b0c02"

def get_size_and_checksum(remote_path, token=None):
    """Look up a file's size and adler32 checksum from dCache instead of computing them locally."""
    headers = auth_headers(token) if token else {}
    resp = requests.get(f"{AMSC_NAMESPACE_API}{remote_path}", params={"checksums": "true"}, headers=headers)
    resp.raise_for_status()
    data = resp.json()
    checksums = {c["type"].lower(): c["value"] for c in data.get("checksums", [])}
    return data["size"], checksums

size, checksums = get_size_and_checksum(FDP_LOCATION, token)

mc.declare_file(
    namespace=NAMESPACE,
    name=FILE_TO_DECLARE,
    dataset_did=f"{NAMESPACE}:{DATASET}",
    size=size,
    checksums=checksums,
    metadata={
        "AmSC.common.type": "artifact",
        "AmSC.common.description": f"File {FILE_TO_DECLARE} from the {DATASET} dataset",
        "AmSC.common.location": f"globus://{FDP_GLOBUS_UUID}{FDP_LOCATION}",
        "fn.locations": [
            f"{AMSC_WEBDAV}{FDP_LOCATION}",
            f"root://amsc.fnal.gov{FDP_LOCATION}",
            f"globus://{FDP_GLOBUS_UUID}{FDP_LOCATION}",
        ],
    },
)

### Update a file's metadata

Modify the metadata of a file that's already been declared. `replace=False` merges the new fields into the existing metadata instead of overwriting the whole record.

In [ ]:
UPDATED_METADATA = 
# UPDATED_METADATA = {"AmSC.common.description": "Updated description"}

def update_file_metadata(namespace, filename, metadata):
    did = f"{namespace}:{filename}"
    mc.update_file(did, replace=False, metadata=metadata)

update_file_metadata(NAMESPACE, FILE_TO_DECLARE, UPDATED_METADATA)
show_file_metadata(f"{NAMESPACE}:{FILE_TO_DECLARE}")